<a href="https://colab.research.google.com/github/i-wild/Python_Ejercicios/blob/master/Eliminar_espacios_dobles_en_excel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
from google.colab import files
import pandas as pd
import re
import os
import sys
import subprocess

def asegurar_dependencia(nombre_pip, nombre_import=None):
    if nombre_import is None:
        nombre_import = nombre_pip.replace("-", "_")
    try:
        __import__(nombre_import)
    except ModuleNotFoundError:
        print(f"Instalando dependencia faltante: {nombre_pip} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", nombre_pip])

asegurar_dependencia("python-calamine", "python_calamine")
asegurar_dependencia("openpyxl")

def limpiar_texto(valor):
    if pd.isna(valor):
        return ""
    if isinstance(valor, str):
        return re.sub(r" {2,}", " ", valor).strip()
    return valor

def normalizar_para_comparacion(valor):
    if pd.isna(valor):
        return ""
    if isinstance(valor, str):
        return re.sub(r" {2,}", " ", valor).strip().casefold()
    return valor

def columna_excel_a_indice(col):
    col = col.strip().upper()
    if not col.isalpha():
        raise ValueError(f"Columna invalida: {col}")
    indice = 0
    for ch in col:
        indice = indice * 26 + (ord(ch) - ord("A") + 1)
    return indice - 1

def indice_a_columna_excel(indice):
    indice += 1
    resultado = ""
    while indice > 0:
        indice, residuo = divmod(indice - 1, 26)
        resultado = chr(65 + residuo) + resultado
    return resultado

def pedir_columnas_a_ignorar():
    print("\nOpcional: puedes ignorar columnas al buscar duplicados.")
    print("Ejemplos validos:")
    print("  I")
    print("  I,K")
    print("  I, K, AA")
    print("Si no quieres ignorar ninguna columna, solo presiona Enter.\n")

    texto = input("Columnas a ignorar: ").strip().upper()

    if not texto:
        print("No se ignorara ninguna columna.")
        return []

    partes = [p.strip() for p in texto.split(",") if p.strip()]
    columnas = []
    for parte in partes:
        columnas.append(columna_excel_a_indice(parte))

    columnas = sorted(set(columnas))
    letras = [indice_a_columna_excel(i) for i in columnas]

    confirmacion = input(
        f"Confirmar columnas a ignorar: {', '.join(letras)} ? (S/N): "
    ).strip().upper()

    if confirmacion != "S":
        print("Operacion cancelada por el usuario antes de procesar.")
        return None

    return columnas

def procesar_excel():
    uploaded = files.upload()

    if not uploaded:
        print("Operacion cancelada: no se selecciono ningun archivo.")
        return

    input_file = next(iter(uploaded.keys()))
    print(f"Archivo recibido: {input_file}")

    extension = os.path.splitext(input_file)[1].lower()
    if extension not in [".xlsx", ".xls"]:
        print("Formato no soportado. Usa un archivo .xlsx o .xls")
        return

    columnas_ignoradas = pedir_columnas_a_ignorar()
    if columnas_ignoradas is None:
        return

    hojas = pd.read_excel(
        input_file,
        sheet_name=None,
        header=None,
        dtype=object,
        engine="calamine"
    )

    resultado = {}
    reporte = []
    letras_ignoradas = [indice_a_columna_excel(i) for i in columnas_ignoradas]

    for nombre_hoja, df in hojas.items():
        filas_originales = len(df)

        # Limpieza del contenido final
        df_limpio = df.apply(lambda col: col.map(limpiar_texto))

        # Copia para comparar duplicados
        df_comparacion = df_limpio.apply(lambda col: col.map(normalizar_para_comparacion))

        # Excluir columnas ignoradas solo de la comparacion
        columnas_presentes = list(df_comparacion.columns)
        columnas_para_comparar = [c for c in columnas_presentes if c not in columnas_ignoradas]

        if not columnas_para_comparar:
            mask_duplicadas = pd.Series([False] * len(df_comparacion), index=df_comparacion.index)
            duplicadas_eliminadas = 0
            nota = "No se eliminaron duplicados porque todas las columnas quedaron ignoradas."
        else:
            df_base_comparacion = df_comparacion[columnas_para_comparar]
            mask_duplicadas = df_base_comparacion.duplicated(keep="first")
            duplicadas_eliminadas = int(mask_duplicadas.sum())
            nota = ""

        df_final = df_limpio.loc[~mask_duplicadas].copy()
        resultado[nombre_hoja] = df_final

        mensaje = (
            f"Hoja '{nombre_hoja}': {filas_originales} -> {len(df_final)} filas "
            f"({duplicadas_eliminadas} duplicadas eliminadas)"
        )
        if nota:
            mensaje += f" | {nota}"
        reporte.append(mensaje)

    base, _ = os.path.splitext(input_file)
    output_file = f"{base}_limpio_sin_duplicados.xlsx"

    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
        for nombre_hoja, df_final in resultado.items():
            df_final.to_excel(
                writer,
                sheet_name=nombre_hoja[:31],
                header=False,
                index=False
            )

    print("\nProceso completado.\n")
    if letras_ignoradas:
        print("Columnas ignoradas para comparar duplicados:", ", ".join(letras_ignoradas))
    else:
        print("No se ignoraron columnas.")

    print()
    for linea in reporte:
        print(linea)

    print(f"\nArchivo generado: {output_file}")
    files.download(output_file)

procesar_excel()


Saving Contactos 20260427.xlsx to Contactos 20260427 (2).xlsx
Archivo recibido: Contactos 20260427 (2).xlsx

Opcional: puedes ignorar columnas al buscar duplicados.
Ejemplos validos:
  I
  I,K
  I, K, AA
Si no quieres ignorar ninguna columna, solo presiona Enter.

Columnas a ignorar: i
Confirmar columnas a ignorar: I ? (S/N): s

Proceso completado.

Columnas ignoradas para comparar duplicados: I

Hoja 'Datos': 453 -> 268 filas (185 duplicadas eliminadas)
Hoja 'Parámetros': 3 -> 3 filas (0 duplicadas eliminadas)

Archivo generado: Contactos 20260427 (2)_limpio_sin_duplicados.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>